<a href="https://colab.research.google.com/github/vishnuofficial004-ui/AI-Powered-ECG-Classification/blob/main/Extraction_and_classification_of_beats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wfdb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from tqdm import tqdm
import numpy as np
import collections
import random
import wfdb
import os


In [ ]:
# Paths
record_base_dir = '/content/drive/MyDrive/MIT_BIH-RECORDS'
output_base_dir = '/content/drive/MyDrive/beats_dataset'

label_folders = {
    'normal': os.path.join(output_base_dir, 'normal'),
    'arrhythmia': os.path.join(output_base_dir, 'arrhythmia'),
    'irregular': os.path.join(output_base_dir, 'irregular'),}

for folder in label_folders.values():
    os.makedirs(folder, exist_ok=True)

NORMAL = ['N','L', 'R','B']
ARRHYTHMIA = ['A', 'a', 'J', 'S', 'V', 'r', 'F', 'E','e', 'j']
IRREGULAR = [ 'Q', '?', 'x', '+', '!','"' ]


In [ ]:
def normalize_beat_segment(beat_segment):
    return (beat_segment - np.mean(beat_segment)) / np.std(beat_segment)


In [ ]:

beat_counts = {'normal': 0, 'arrhythmia': 0, 'irregular': 0}

window_size = 216
record_folders = sorted(os.listdir(record_base_dir))

for folder_name in tqdm(record_folders, desc="Counting Beats"):
    record_path = os.path.join(record_base_dir, folder_name, folder_name)
    try:
        signal, _ = wfdb.rdsamp(record_path, channels=[0])
        annotation = wfdb.rdann(record_path, 'atr')
    except Exception as e:
        print(f"Skipping {folder_name} due to error: {e}")
        continue

    for sym in annotation.symbol:
        if sym in NORMAL:
            beat_counts['normal'] += 1
        elif sym in ARRHYTHMIA:
            beat_counts['arrhythmia'] += 1
        elif sym in IRREGULAR:
            beat_counts['irregular'] += 1

print("Beat counts before balancing:", beat_counts)

# Set limits: keep arrhythmia as is, limit normal to arrhythmia count, irregular will be upsampled later
beat_limits = {
    'normal': beat_counts['arrhythmia'],
    'arrhythmia': None,
    'irregular': None
}


Counting Beats: 100%|██████████| 17/17 [00:23<00:00,  1.35s/it]

Beat counts before balancing: {'normal': 34574, 'arrhythmia': 4083, 'irregular': 892}


In [ ]:

beat_id = 0

saved_counts = {'normal': 0, 'arrhythmia': 0, 'irregular': 0}

# Step 1: Extract arrhythmia and irregular beats, save immediately
for folder_name in tqdm(record_folders, desc="Extracting Arrhythmia and Irregular Beats"):
    record_path = os.path.join(record_base_dir, folder_name, folder_name)
    try:
        signal, _ = wfdb.rdsamp(record_path, channels=[0])
        annotation = wfdb.rdann(record_path, 'atr')
    except Exception as e:
        print(f"Skipping {folder_name} due to error: {e}")
        continue

    signal = signal.flatten()

    for idx, sym in enumerate(annotation.symbol):
        if sym in ARRHYTHMIA or sym in IRREGULAR:
            label = 'arrhythmia' if sym in ARRHYTHMIA else 'irregular'

            sample_loc = annotation.sample[idx]
            start = sample_loc - window_size // 2
            end = sample_loc + window_size // 2
            if start < 0 or end > len(signal):
                continue

            beat_segment = signal[start:end]
            beat_segment = normalize_beat_segment(beat_segment)

            save_path = os.path.join(label_folders[label], f"{label}_{beat_id}.npy")
            np.save(save_path, beat_segment)

            saved_counts[label] += 1
            beat_id += 1

print(f"Arrhythmia and irregular beats saved. Counts: {saved_counts}")

# Step 2: Collect normal beat positions (but do not save yet)
normal_beats_per_record = collections.defaultdict(list)

for folder_name in tqdm(record_folders, desc="Collecting Normal Beats"):
    record_path = os.path.join(record_base_dir, folder_name, folder_name)
    try:
        signal, _ = wfdb.rdsamp(record_path, channels=[0])
        annotation = wfdb.rdann(record_path, 'atr')
    except Exception as e:
        print(f"Skipping {folder_name} due to error: {e}")
        continue

    for idx, sym in enumerate(annotation.symbol):
        if sym in NORMAL:
            sample_loc = annotation.sample[idx]
            normal_beats_per_record[folder_name].append(sample_loc)

print("Collected normal beats per record.")

print("Arrhythmia beats saved:", saved_counts['arrhythmia'])
print("Normal beats limit:", saved_counts['arrhythmia'] )


# Step 3: Balanced saving of normal beats using round-robin approach
normal_limit = saved_counts['arrhythmia']
records_list = list(normal_beats_per_record.items())

signals_cache = {}
for folder_name, _ in records_list:
    record_path = os.path.join(record_base_dir, folder_name, folder_name)
    signal, _ = wfdb.rdsamp(record_path, channels=[0])
    signals_cache[folder_name] = signal.flatten()

normal_saved = 0
record_idx = 0

while normal_saved < normal_limit:
    folder_name, beats_list = records_list[record_idx]
    if not beats_list:
        record_idx = (record_idx + 1) % len(records_list)
        continue

    sample_loc = beats_list.pop(0)
    signal = signals_cache[folder_name]

    start = sample_loc - window_size // 2
    end = sample_loc + window_size // 2
    if start < 0 or end > len(signal):
        record_idx = (record_idx + 1) % len(records_list)
        continue

    beat_segment = signal[start:end]
    beat_segment = normalize_beat_segment(beat_segment)

    save_path = os.path.join(label_folders['normal'], f"normal_{beat_id}.npy")
    np.save(save_path, beat_segment)

    normal_saved += 1
    beat_id += 1
    saved_counts['normal'] += 1

    record_idx = (record_idx + 1) % len(records_list)

print(f"Normal beats balanced and saved from multiple records. Total normal beats saved: {normal_saved}")
print(f"Final saved counts: {saved_counts}")




In [ ]:
[shutil.copy2(os.path.join(src, f), os.path.join(dst, f)) for label in ['normal', 'arrhythmia', 'irregular'] for src, dst in [(f'/content/beats_dataset/{label}', f'/content/drive/MyDrive/beats_dataset/{label}')] if os.path.exists(src) for f in set(os.listdir(src)) - set(os.listdir(dst))]

[]

In [ ]:


# Load all irregular beat files
irregular_files = os.listdir(label_folders['irregular'])
arrhythmia_count = saved_counts['arrhythmia']
irregular_count = saved_counts['irregular']

upsample_needed = arrhythmia_count - irregular_count
print(f"Upsampling irregular beats by: {upsample_needed}")

if upsample_needed > 0:
    for i in range(upsample_needed):
        # Pick a random irregular beat file to duplicate
        file_to_duplicate = random.choice(irregular_files)
        beat_data = np.load(os.path.join(label_folders['irregular'], file_to_duplicate))

        # Save with new file name
        new_id = beat_id + i
        save_path = os.path.join(label_folders['irregular'], f"irregular_{new_id}.npy")
        np.save(save_path, beat_data)

    print(f"Upsampled irregular beats to match arrhythmia count.")
else:
    print("No upsampling needed for irregular beats.")


Upsampling irregular beats by: 3203
Upsampled irregular beats to match arrhythmia count.


In [ ]:
for label, folder in label_folders.items():
    count = len(os.listdir(folder))
    print(f"Final beats count for {label}: {count}")


Final beats count for normal: 4081
Final beats count for arrhythmia: 4081
Final beats count for irregular: 4081


In [ ]:
[shutil.copy2(os.path.join(src, f), os.path.join(dst, f)) for label in ['normal', 'arrhythmia', 'irregular'] for src, dst in [(f'/content/beats_dataset/{label}', f'/content/drive/MyDrive/beats_dataset/{label}')] if os.path.exists(src) for f in set(os.listdir(src)) - set(os.listdir(dst))]


[]